# Exploratory analysis and modeling

OLS regression, Random Forest, K-Means and diagnostics. Run `01_data_preparation.ipynb` first to generate `data/processed/nm_county_clean.csv`. Original analytical methods retained.

In [ ]:
from pathlib import Path

# Run from the repository root or its src/ folder.
# If using Colab, upload/open the repository files in a runtime where
# the data/ directory is available; no personal Google Drive path is required.
candidates = [Path.cwd(), Path.cwd().parent, Path.cwd() / "NM-Mental-Health-Housing-Stability"]
project_dir = next((p.resolve() for p in candidates if (p / "data" / "raw").is_dir()), None)
if project_dir is None:
    raise FileNotFoundError(
        "Could not find data/raw. Run from the repository root or src/ folder "
        "with the project's data/ directory available."
    )
%cd {project_dir}
print("Project:", project_dir)


## Imports

In [ ]:
import os
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

from statsmodels.stats.outliers_influence import variance_inflation_factor
import statsmodels.api as sm

import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
nm_subset = pd.read_csv('data/processed/nm_county_clean.csv')
print('Loaded clean county data:', nm_subset.shape)
display(nm_subset.head())

## EDA

In [ ]:
# identify potential outliers
for col in nm_subset.columns[1:]:
    print(f"\n{col}")
    print("Min:")
    print(nm_subset.loc[nm_subset[col].idxmin(), ['Name', col]])

    print("\nMax:")
    print(nm_subset.loc[nm_subset[col].idxmax(), ['Name', col]])

### Observations - outliers

Harding County has 0 mental health providers
---
Los Alamos appears repeatedly:

| Variable |	Value
| -------- | --------
Poor Mental Health Days |	Lowest
Children in Poverty |	Lowest
Unemployment |	Lowest
Income Inequality |	Lowest
Child Care Cost Burden |	Lowest
Median Household Income |	Highest ($146,208)
---

McKinley County is on the opposite end of the spectrum:
| Variable                | Value   |
| ----------------------- | ------- |
| Poor Mental Health Days | Highest |
| Children in Poverty     | Highest |
| Income Inequality       | Highest |
---
Taos County:
|Variable | Value
|-------- | ---------
|Severe Housing Cost Burden | Highest
|Childcare Cost Burden | Highest

In [ ]:
# investigage the counties with very low provider values
nm_subset[
    nm_subset['Mental Health Providers raw value'] <= 0.001
][['Name', 'Mental Health Providers raw value']]


In [ ]:
# correlation matrix
corr = nm_subset.drop(columns='Name').corr().round(3)
corr

### Observations - correlations
Severe Housing cost Burden vs Mental Health Providers
- r = 0.307
- weak positive correlation
- somewhat surprising (we may have expected a negative correlation)
- urban counties may have more providers AND more expensive housing
- i.e. provider availability may be acting as a proxy for urbanization
---
Severe Housing Cost Burden vs Poor Mental Health Days
- r = -0.037
- almost no correlation
- counties with worse mental health do not appear to have substantially higher housing cost burden
---
Variable most related to Housing Cost Burden
- Child Care Cost Burden r = 0.350
- childcare and housing costs are likely two of the biggest strains on household budgets


### Possible multicollinearity
Poverty vs Median Income
- r = -0.861
- strong negative correlation
---
Poverty vs Poor Mental Health Days
- r = 0.757
- strong positive correlation

---
Will need to check variance inflation factors (VIF's) before fitting the final regression

In [ ]:
# scatter plot of relationships
plt.figure(figsize=(8,5))

plt.scatter(
    nm_subset['Mental Health Providers raw value'],
    nm_subset['Severe Housing Cost Burden raw value']
)

plt.xlabel('Mental Health Providers')
plt.ylabel('Severe Housing Cost Burden')
plt.title('Mental Health Providers vs Housing Cost Burden')

plt.show()

In [ ]:
# the same plot with county names
plt.figure(figsize=(8,5))

plt.scatter(
    nm_subset['Mental Health Providers raw value'],
    nm_subset['Severe Housing Cost Burden raw value']
)

for i, row in nm_subset.iterrows():
    plt.annotate(
        row['Name'].replace(' County',''),
        (
            row['Mental Health Providers raw value'],
            row['Severe Housing Cost Burden raw value']
        ),
        fontsize=8
    )

plt.xlabel('Mental Health Providers')
plt.ylabel('Severe Housing Cost Burden')
plt.title('Mental Health Providers vs Housing Cost Burden')
plt.show()

### Observations - mental health providers vs housing cost burden
Mental Healthcare Providers raw value = Number of mental health care providers per 100,000 population
- There is a cluster of counties between 0.001 and 0.006 mental healthcare providers
- A few counties have much higher provider values
- housing burden is spread more widely within each provider range
- Guadalupe and quay counties are are far right of every other county
- Quay county is also high on housing burden
De Baca county has high provider access and low housing burden
- Taos, Santa Fe, and Bernalillo counties have relative high housing cost burden and provider access
  - hypothesis: counties with more providers tend to be more urban, which also tend to have more expensive housing
    - there could be a confounding factor

### Conclusions so far
- mental health providers is probablly not sufficient to explain housing burden
- after controlling for poverty, income, unemployment, and childcare burden, does provider access still matter?
- providers vs housing burden turned out to be more complicated than expected
---
There are 2 different mental health concepts here:
  1. mental health access (providers)
  1. mental health need (poor mental health days)

In [ ]:
# examine poor mental health days vs housing cost burden
plt.figure(figsize=(8,5))

plt.scatter(
    nm_subset['Poor Mental Health Days raw value'],
    nm_subset['Severe Housing Cost Burden raw value']
)

plt.xlabel('Poor Mental Health Days')
plt.ylabel('Severe Housing Cost Burden')
plt.title('Poor Mental Health Days vs Housing Cost Burden')

plt.show()

### Observations - poor mental health days vs housing cost burden
- there isn't much of a relationship between poor mental health days and greater housing instability (as measured by severe housing cost burden)
- possible reasons
  - housing burden is driven more by housing market conditions
  - poor mental health days is measureing something different than access
  - socioecononmic variables are confounding the relationship
  - may need to do multivariate regression

In [ ]:
# correlation heat map
plt.figure(figsize=(10,8))

sns.heatmap(
    corr,
    annot=True,
    cmap='coolwarm',
    center=0,
    fmt='.2f'
)

plt.title('Correlation Matrix')
plt.show()

### Observations - correlation heatmap
- children in poverty and median household income are strongly (negatively) correlated
- poor mental health days is acting like a socioeconomic indicator (similar to poverty or median household income)
  - this suggests that counties with more poor mental health days tend to be poorer counties
- mental health providers is not strongly correlated to socioeconomic indicators like poverty or income

### VIF definition
"The variance inflation factor quantifies the degree to which the variation of an independent variable increases the standard error in multiple regression models, helping to identify multicollinearity issues.

- VIF equal to 1 = variables are not correlated
- VIF between 1 and 5 = variables are moderately correlated
- VIF greater than 5 = variables are highly correlated

The higher the VIF, the higher the possibility that multicollinearity exists, and further research is required. When VIF is higher than 10, there is significant multicollinearity that needs to be corrected."

Reference:
- https://www.investopedia.com/terms/v/variance-inflation-factor.asp

- https://online.stat.psu.edu/stat462/node/180/


**Methodology note:** The historical VIF code below does not include an intercept, so its values may be inflated. Recalculate with `sm.add_constant(X)` before interpreting multicollinearity; the original model fits are unchanged.

In [ ]:
# check VIF table
X = nm_subset.drop(
    columns=[
        'Name',
        'Severe Housing Cost Burden raw value'
    ]
)

vif = pd.DataFrame()
vif["Variable"] = X.columns
vif["VIF"] = [
    variance_inflation_factor(X.values, i)
    for i in range(X.shape[1])
]

vif

### Observations - VIF table
- most of the VIF values are high
- median income, poor mental health days, and unemployment are likely all measuring the same socioeconomic disadvantage indicator
- with only 33 counties, the overlap is magnified
- the mental health providers variable is relatively independent of the other predictors

### Reduce predictor set

In [ ]:
# choose only one of the socioeconomic disadvantage indicator predictors
nm_subset2 = nm_subset[[
    'Mental Health Providers raw value',
    'Poor Mental Health Days raw value',
    'Children in Poverty raw value',
    'Median Household Income raw value',
]]
nm_subset2.head()

### Features data dictionary

| Feature | Definition
| ------- | -----------
| Mental Health Providers raw value | Number of mental health care providers per 100,000 population
| Poor Mental Health Days raw value | Average number of mentally unhealthy days reported in past 30 days (age-adjusted)
| Children in Poverty raw value | Percentage of people under age 18 in poverty
| Median Household Income raw value | Median household income
| Childcare Cost Burden | Child care costs for a household with two children as a percent of median household income


In [ ]:
# calculate VIF's with reduced set
X_reduced = nm_subset[[
    'Mental Health Providers raw value',
    'Poor Mental Health Days raw value',
    'Children in Poverty raw value',
    'Median Household Income raw value'
]]

vif_reduced = pd.DataFrame()
vif_reduced['Variable'] = X_reduced.columns
vif_reduced['VIF'] = [
    variance_inflation_factor(X_reduced.values, i)
    for i in range(X_reduced.shape[1])
]

vif_reduced

### Observations - reduced set
The last 3 predictors are likely all measuring overlapping aspects of county socioeconomic disadvantage

In [ ]:
# look at predictor correlations onlly
X_reduced.corr().round(3)

## Modeling - OLS

### Baseline OLS model
 OLS model is the standard method for performing linear regression. It estimates the relationship between a dependent variable and one or more independent variables by finding the "line of best fit" that minimizes the sum of the squared differences (residuals) between the actual and predicted values.

Features:
- Mental Health Providers raw value
- Poor Mental Health Days raw value

Assumptions:
- linear relationships
- independent observations (no correlation between errors)
- homoscedasticity - variance of error terms must be constant across all values of the independent variables
- no multicollinearity

In [ ]:
# fit the baseline OLS
X = nm_subset[[
    'Mental Health Providers raw value',
    'Poor Mental Health Days raw value'
]]

X = sm.add_constant(X)

y = nm_subset['Severe Housing Cost Burden raw value']

model_1 = sm.OLS(y, X).fit()

print(model_1.summary())

### Interpretation
R-squared = .095
- the two features only explain 9.5% of the variation in housing burden across NM counties
- adjusted for model complexity (only 2 predictors), R-squared is even lower 0.034
- this model doesn't capture most of what drives housing burden
- Prob(f-statistic): p = 0.225
  - p should be less than 0.05 for the model to be statistically significant, this one isn't

Summary:
A baseline OLS regression found no statistically significant association between county-level poor mental health days and severe housing cost burden. Mental health provider availability exhibited a positive but non-significant association with housing burden (β = 2.75, p = 0.089). Overall, the model explained approximately 9.5% of the variance in housing burden across New Mexico counties.

Reference:
- https://online.stat.psu.edu/stat462/node/93/


### Model 2 OLS
- add Median Household Income raw value

In [ ]:
X = nm_subset[[
    'Mental Health Providers raw value',
    'Poor Mental Health Days raw value',
    'Median Household Income raw value',
]]

X = sm.add_constant(X)

y = nm_subset['Severe Housing Cost Burden raw value']

model_2 = sm.OLS(y, X).fit()

print(model_2.summary())

### Model 3 OLS
-  Child Care Cost Burden instead of Median Household Income raw value

In [ ]:
X = nm_subset[[
    'Mental Health Providers raw value',
    'Poor Mental Health Days raw value',
    'Child Care Cost Burden raw value',
]]

X = sm.add_constant(X)

y = nm_subset['Severe Housing Cost Burden raw value']

model_3 = sm.OLS(y, X).fit()

print(model_3.summary())

### Model 3 OLS
- mental health providers
- poor mental health days
- child care cost burden

In [ ]:
# does child care cost burden and mental health providers have multicollinearity?
X_model3 = nm_subset[[
    'Mental Health Providers raw value',
    'Poor Mental Health Days raw value',
    'Child Care Cost Burden raw value'
]]

vif = pd.DataFrame()
vif["Variable"] = X_model3.columns
vif["VIF"] = [
    variance_inflation_factor(X_model3.values, i)
    for i in range(X_model3.shape[1])
]

vif

### Compare the 3 models
| Metric | Model 1 | Model 2 | Model 3 |
| ------ | ------- | ------- | ------- |
| Predictors | Providers + MH Days | + Income | + Child Care Burden |
| R²| 0.095 | 0.111 | **0.328** |
| Adjusted R² | 0.034 | 0.020 | **0.259** |
| Model p-value | 0.225 | 0.323 | **0.008** |

Model 3 is the strongest so far
- it is statistically significant
- after accounting for child care cost burden, counties with greater mental health provider availability tend to have higher severe housing cost burden
  - this could mean that urban areas have more mental health providers as well as higher housing cost burden
  - this relationship is statistically significant
- counties with higher child care cost burdens tend to have higher housing cost burdens
- Poor Mental Health Days is not statistically significant in all 3 models. It does not seem to be a strong predictor of severe housing cost burden at the county level in NM

A multiple linear regression model including mental health provider availability, poor mental health days, and child care cost burden explained 32.8% of the variation in severe housing cost burden across New Mexico counties (R² = 0.328, p = 0.008). Mental health provider availability (β = 3.87, p = 0.010) and child care cost burden (β = 0.41, p = 0.004) were significant positive predictors of housing burden, while poor mental health days was not statistically significant (p = 0.233).


### Model 4 OLS
Question:

After accounting for household financial pressure (child care burden), is mental health provider availability still associated with housing cost burden?

X:
- mental health providers
- child care cost burden

Y:
- severe housing cost burden

In [ ]:
# fit model 4
X = nm_subset[[
    'Mental Health Providers raw value',
    'Child Care Cost Burden raw value'
]]

X = sm.add_constant(X)

y = nm_subset['Severe Housing Cost Burden raw value']

model_4 = sm.OLS(y, X).fit()

print(model_4.summary())

### Model 4 Interpretation
Counties with greater mental health provider availability and higher child care cost burden tended to exhibit higher levels of severe housing cost burden. Together, these variables explained approximately 29% of the variation in housing burden across New Mexico counties (R² = 0.294, p = 0.005).

### Check for counties that may have a disproportionate influence on the regression

In [ ]:
# Cook's distance plot to see if any points have a large prediction error
influence = model_4.get_influence()

cooks = influence.cooks_distance[0]

plt.figure(figsize=(10,5))
plt.stem(range(len(cooks)), cooks)

plt.axhline(
    y=4/len(cooks),
    color='red',
    linestyle='--',
    label=f'4/n = {4/len(cooks):.3f}'
)

plt.title("Cook's Distance")
plt.xlabel("Observation")
plt.ylabel("Cook's Distance")
plt.legend()

plt.show()

In [ ]:
# list the counties in order of largest to smallest Cook's distance
cooks_df = nm_subset.copy()
cooks_df["cooks_distance"] = cooks

cooks_df[["Name", "cooks_distance"]].sort_values(
    by="cooks_distance",
    ascending=False
).head(10)

### Model 5 OLS - sensitivity analysis

In [ ]:
# sensitivity analysis: if we remove the 2 most influential counties, does the provider effect disappear?
sensitivity_df = nm_subset[
    ~nm_subset['Name'].isin([
        'DeBaca County',
        'Guadalupe County'
    ])
].copy()

X = sensitivity_df[[
    'Mental Health Providers raw value',
    'Child Care Cost Burden raw value'
]]

X = sm.add_constant(X)

y = sensitivity_df['Severe Housing Cost Burden raw value']

sensitivity_model = sm.OLS(y, X).fit()

print(sensitivity_model.summary())

### Compare Sensitivity Analysis model with primary model

Primary model:
- R² = 0.294
- Providers p = 0.011
- Child Care p = 0.007

Sensitivity Analysis (excluding influential counties):
- R² = 0.466
- Providers p < 0.001
- Child Care p = 0.015

The positive association between mental health provider availability and severe housing cost burden remained statistically significant and became stronger after excluding the two most influential counties identified through Cook's Distance diagnostics.

Observations:
- There is no meaningful evidence of a relationship between poor mental health days and housing cost burden
- There is a positive association between mental health provider availability and housing cost burden
- Child care cost burden has a statistically significant relationship with housing burden



## Modeling - Random Forest

In [ ]:
# fit random forest model
X = nm_subset[[
    'Mental Health Providers raw value',
    'Poor Mental Health Days raw value',
    'Children in Poverty raw value',
    'Unemployment raw value',
    'Income Inequality raw value',
    'Child Care Cost Burden raw value',
    'Median Household Income raw value'
]]

y = nm_subset['Severe Housing Cost Burden raw value']

rf = RandomForestRegressor(
    n_estimators=500,
    random_state=42
)

rf.fit(X, y)

y_pred = rf.predict(X)

print("R²:", r2_score(y, y_pred))

feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': rf.feature_importances_
}).sort_values(
    by='Importance',
    ascending=False
)

feature_importance

Observations - random forest
- R-squared = 0.870
- training R-squared

The random forest model explained about about 87% of the variation in housing burden **in the training data**.

- the most important and second most important features are mental health providers and child care cost burden, similar to the significance in the OLS model
- poor mental health days was not significant in the OLS model, but it does have some importance in the random forest model

In [ ]:
# random forest feature importance plot
feature_importance = feature_importance.sort_values(
    by='Importance'
)

plt.figure(figsize=(8,5))

plt.barh(
    feature_importance['Feature'],
    feature_importance['Importance']
)

plt.xlabel('Importance')
plt.title('Random Forest Feature Importance')

plt.show()

OLS asked:

Which variables are associated with housing burden?

Random Forest asked:

Which variables are most important?

K-Means asks:

Are there distinct types of counties that might benefit from different policy approaches?

## Modeling -  K-means Clustering

In [ ]:
# choose features
cluster_features = nm_subset[[
    'Mental Health Providers raw value',
    'Poor Mental Health Days raw value',
    'Severe Housing Cost Burden raw value',
    'Children in Poverty raw value',
    'Median Household Income raw value'
]]

scaler = StandardScaler()

X_scaled = scaler.fit_transform(cluster_features)

print(X_scaled.shape)

In [ ]:
# determine the appropriate number of clusters (k) with the elbow method
# (the elbow is where the line starts to flatten)
# fit model with k = 1-10 and look at the inertia (within-cluster sum of squares)

inertia = []

for k in range(1, 11):
    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    kmeans.fit(X_scaled)

    inertia.append(kmeans.inertia_)

plt.figure(figsize=(8,5))

plt.plot(
    range(1,11),
    inertia,
    marker='o'
)

plt.xlabel('Number of Clusters (k)')
plt.ylabel('Inertia')
plt.title('Elbow Method for K-Means')

plt.show()

In [ ]:
# fit the model
kmeans = KMeans(
    n_clusters=4,
    random_state=42,
    n_init=10
)

nm_subset['Cluster'] = kmeans.fit_predict(X_scaled)

nm_subset[['Name', 'Cluster']].head()

In [ ]:
# how many counties are in each cluster
nm_subset['Cluster'].value_counts().sort_index()

In [ ]:
# get information on the county that makes up its own cluster
nm_subset[nm_subset['Cluster'] == 2][
    ['Name',
     'Mental Health Providers raw value',
     'Poor Mental Health Days raw value',
     'Severe Housing Cost Burden raw value',
     'Children in Poverty raw value',
     'Median Household Income raw value']
]

### Observation - Los Alamos County
Los Alamos county has the lowest poor mental health days, the lowest children in poverty, a very low housing cost burden, and at least triple the median household income compared to the rest of the state ($146,208).

In [ ]:
# create a cluster summary table
cluster_summary = nm_subset.groupby('Cluster')[
    [
        'Mental Health Providers raw value',
        'Poor Mental Health Days raw value',
        'Severe Housing Cost Burden raw value',
        'Children in Poverty raw value',
        'Median Household Income raw value'
    ]
].mean().round(3)

cluster_summary

### Cluster Summary
|  | Cluster 0 | Cluster 1 | Cluster 2 | Cluster 3 |
| -------- | --------- | -------- | -------- | -------- |
Providers
Poor Mental Health Days
Housing Burden
Poverty
Median Income |

### Observations - Cluster Summary Table
- In the OLS model, there was a positive association between mental health providers and housing burden
- In the K-means clustering model, we see that the counties with more providers usually belong to clusters that also have higher housing burden, larger population centers, and more developed infrastructures

- Provider availability may be a marker of county development and services, rather than causing housing burden
---


## Basic Analysis of the 3 Models

OLS Regression

Answers:
- Which factors are statistically associated with housing burden?

Finding:
- Mental Health Providers
- Child Care Cost Burden
---

Random Forest

Answers:
- Which variables are most important?

Finding:
- Mental Health Providers
- Child Care Cost Burden
---

K-Means Clustering

Answers:
- Are there distinct county types?

Finding:
- Four county profiles
- One unique outlier (Los Alamos)
- Evidence that housing burden and provider access cluster geographically and economically